In [1]:
import pandas as pd
import os
import sys
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from csbdeep.utils import normalize
from skimage.filters import median, gaussian
from skimage.morphology import disk
import cv2

In [2]:
annotation_csv = pd.read_excel('/mnt/d/lding/CLS/mousumiLiuDinner/selected_for_annotation_mt_strcture_20250310_MA_batch1.xlsx')
cellpose_results_csv = pd.read_csv('/mnt/d/lding/CLS/mousumiLiuDinner/guv_mt_cell_df_circle_selected_std15_annotated_batch1.csv')
circle_results_csv = pd.read_csv('/mnt/d/lding/CLS/mousumiLiuDinner/guv_mt_cell_df_circle_selected_std13_20250515_B_batch2.csv')

In [3]:
import numpy as np
from scipy.ndimage import map_coordinates

def circular_to_square(image, cx, cy, r, output_size=128):
    yy, xx = np.meshgrid(np.linspace(-1, 1, output_size), np.linspace(-1, 1, output_size))
    
    # (xx, yy) covers square [-1, 1] × [-1, 1], map this to disk:
    mask = xx**2 + yy**2 <= 1

    # Map unit disk to image coordinates
    x_coords = cx + xx * r
    y_coords = cy + yy * r

    # Flatten coordinates
    coords = np.vstack([y_coords.ravel(), x_coords.ravel()])
    
    # Sample using bilinear interpolation
    sampled = map_coordinates(image, coords, order=1, mode='constant', cval=0)
    
    # Reshape back and apply circular mask
    square_image = sampled.reshape((output_size, output_size))
    square_image[~mask] = 0  # Optional: mask out corners
    
    return square_image


In [4]:
import numpy as np
from scipy.ndimage import map_coordinates

def disk_to_square(image, cx, cy, r, output_size=128):
    """
    Warp a circular region (disk) into a square of shape (output_size, output_size)
    """
    image = image.astype(np.float32)
    
    # Create normalized square grid: x, y ∈ [-1, 1]
    lin = np.linspace(-1, 1, output_size)
    xv, yv = np.meshgrid(lin, lin)

    # Map square to disk using a conformal mapping (polar-style)
    r_norm = np.sqrt(xv**2 + yv**2)
    theta = np.arctan2(yv, xv)

    # Normalized radial stretch: remap square → disk
    denominator = np.maximum(np.abs(np.cos(theta)), np.abs(np.sin(theta)))
    denominator[denominator == 0] = 1e-6  # avoid divide by zero
    x_disk = (r_norm * denominator) * np.cos(theta)
    y_disk = (r_norm * denominator) * np.sin(theta)

    # Scale and shift to actual image coordinates
    x_mapped = cx + x_disk * r
    y_mapped = cy + y_disk * r

    # Sample from image
    coords = np.vstack([y_mapped.ravel(), x_mapped.ravel()])
    sampled = map_coordinates(image, coords, order=1, mode='constant', cval=0)

    # Reshape and mask
    square = sampled.reshape((output_size, output_size))    

    edge_intensity = np.concatenate([square[0, :], square[-1, :]])
    edge_intensity = edge_intensity[edge_intensity > 0]

    fill_value = np.mean(edge_intensity) if edge_intensity.size > 0 else 0
    square[square == 0] = fill_value

    return square


In [5]:
import numpy as np

def crop_circle_detour(image, cx, cy, r, output_size=128):
    """
    Crop a circular region centered at (cx, cy) with radius r
    into a fixed-size output using a safe double-crop-and-center detour.
    No rescaling or warping is done.
    """
    image = image.astype(np.float32)
    cx, cy = int(round(cx)), int(round(cy))
    temp_size = output_size * 2
    half_temp = temp_size // 2
    half_out = output_size // 2

    # Step 1: get bounds for temp crop
    x_min = cx - half_temp
    x_max = cx + half_temp
    y_min = cy - half_temp
    y_max = cy + half_temp

    # Step 2: create padded temp canvas
    temp_crop = np.zeros((temp_size, temp_size), dtype=np.float32)

    # Compute valid ranges
    src_x_min = max(x_min, 0)
    src_x_max = min(x_max, image.shape[1])
    src_y_min = max(y_min, 0)
    src_y_max = min(y_max, image.shape[0])

    dst_x_min = src_x_min - x_min
    dst_x_max = dst_x_min + (src_x_max - src_x_min)
    dst_y_min = src_y_min - y_min
    dst_y_max = dst_y_min + (src_y_max - src_y_min)

    # Fill valid region
    temp_crop[dst_y_min:dst_y_max, dst_x_min:dst_x_max] = image[src_y_min:src_y_max, src_x_min:src_x_max]

    # Step 3: extract central output crop
    center_crop = temp_crop[half_temp - half_out:half_temp + half_out, half_temp - half_out:half_temp + half_out]

    # Step 4: circular masking
    yy, xx = np.ogrid[:output_size, :output_size]
    center = output_size // 2
    safe_r = min(r, center)
    mask = (xx - center) ** 2 + (yy - center) ** 2 <= safe_r ** 2
    small_mask = (xx - center) ** 2 + (yy - center) ** 2 <= int(0.9 * safe_r) ** 2
    ring_mask = mask & ~small_mask

    ring_values = center_crop[ring_mask]
    fill_value = np.mean(ring_values) if ring_values.size > 0 else 0
    center_crop[~mask] = fill_value
    center_crop[center_crop==0] = fill_value
    
    return center_crop


In [6]:
import numpy as np
from scipy.ndimage import map_coordinates

def crop_rescale_circle(image, cx, cy, r, output_size=128):
    """
    Warp a circular region (disk) into a square of shape (output_size, output_size)
    """
    image = image.astype(np.float32)
    
    # Create normalized square grid: x, y ∈ [-1, 1]
    lin = np.linspace(-1, 1, output_size)
    xv, yv = np.meshgrid(lin, lin)

    # Disk mask: keep points inside the unit circle
    mask = (xv**2 + yv**2) <= 1.0
    small_mask = (xv**2 + yv**2) <= 0.9

    # Map square to disk using a conformal mapping (polar-style)
    r_norm = np.sqrt(xv**2 + yv**2)
    theta = np.arctan2(yv, xv)

    # Normalized radial stretch: remap square → disk
    x_disk = r_norm * np.cos(theta)
    y_disk = r_norm * np.sin(theta)

    # Scale and shift to actual image coordinates
    x_mapped = cx + x_disk * r
    y_mapped = cy + y_disk * r

    # Sample from image
    coords = np.vstack([y_mapped.ravel(), x_mapped.ravel()])
    sampled = map_coordinates(image, coords, order=1, mode='constant', cval=0)

    # Reshape and mask
    square = sampled.reshape((output_size, output_size))

    ring_mask = mask & ~small_mask
    intsity_array = square[ring_mask>0]
    
    square[~mask] = np.mean(intsity_array)  # Optional: mask the corners outside the disk
    square[square==0] = np.mean(intsity_array)
    return square


In [7]:
mt_99p = 486
mt_1p = 139
guv_99p = 520
guv_1p = 118
sigma = 1

In [8]:
output_folder = '/mnt/d/lding/CLS/mousumiLiuDinner/set1to5_processed_results/Microtubule_GUV-Liu-20250106T211105Z-001/processed_GUV_MT/from_annotation_to_circle_square_edge'

annotated_circle_detection_png_folder = os.path.join(output_folder, 'MT_circle_detection_png')
annotated_circle_detection_tiff_folder = os.path.join(output_folder, 'MT_circle_detection_tiff')
annotated_circle_square_tiff_folder = os.path.join(output_folder, 'MT_square_image')
annotated_circle_square_panel_folder = os.path.join(output_folder, 'panel')
annotated_MT_circle_crop_tiff_folder = os.path.join(output_folder, 'MT_circle_crop_tiff')
annotated_MT_rescaled_circle_crop_tiff_folder = os.path.join(output_folder, 'MT_rescaled_circle_crop_tiff')

os.makedirs(annotated_circle_detection_png_folder,exist_ok=True)
os.makedirs(annotated_circle_detection_tiff_folder,exist_ok=True)
os.makedirs(annotated_circle_square_tiff_folder,exist_ok=True)
os.makedirs(annotated_circle_square_panel_folder,exist_ok=True)
os.makedirs(annotated_MT_circle_crop_tiff_folder,exist_ok=True)
os.makedirs(annotated_MT_rescaled_circle_crop_tiff_folder,exist_ok=True)


In [9]:
for index, row in annotation_csv.iterrows():
    patch_tif_filename = row['Filename'].replace('.png', '.tif').replace("=", "")
    
    # Match in cellpose_results_csv
    matching_rows_batch1 = cellpose_results_csv[
        cellpose_results_csv['obj_filename']==patch_tif_filename]

    if matching_rows_batch1.empty:
        print(patch_tif_filename)

In [10]:
counts = 0

for index, row in annotation_csv.iterrows():
    patch_tif_filename = row['Filename'].replace('.png', '.tif').replace("=", "")
    
    # Match in cellpose_results_csv
    matching_rows_batch1 = cellpose_results_csv[
        cellpose_results_csv['obj_filename']==patch_tif_filename]

    if not matching_rows_batch1.empty:
        obj_folder = matching_rows_batch1.iloc[0]['obj_folder']
        obj_filename =  matching_rows_batch1.iloc[0]['obj_filename']
        GUV_folder_path = matching_rows_batch1.iloc[0]['GUV_folder_path']
        GUV_file_name = matching_rows_batch1.iloc[0]['GUV_file_name']
        MT_folder_path  = matching_rows_batch1.iloc[0]['MT_folder_path']
        MT_file_name = matching_rows_batch1.iloc[0]['MT_file_name']
        condition = matching_rows_batch1.iloc[0]['condition']
        cx = matching_rows_batch1.iloc[0]['cx']
        cy = matching_rows_batch1.iloc[0]['cy']
        diameter = matching_rows_batch1.iloc[0]['diameter']
        date = matching_rows_batch1.iloc[0]['date']

        cell_ID = matching_rows_batch1.iloc[0]['cell_ID']
        cell_index = matching_rows_batch1.iloc[0]['cell_index']


        bbox_1  = matching_rows_batch1.iloc[0]['bbox_1']
        bbox_2  = matching_rows_batch1.iloc[0]['bbox_2']
        bbox_3  = matching_rows_batch1.iloc[0]['bbox_3']
        bbox_4  = matching_rows_batch1.iloc[0]['bbox_4']


        GUV_image = tifffile.imread(os.path.join(GUV_folder_path,GUV_file_name))
        MT_image = tifffile.imread(os.path.join(MT_folder_path,MT_file_name))
        cellpose_crop_MT_image = tifffile.imread(os.path.join(obj_folder,obj_filename))

        if GUV_image.ndim ==3:
            GUV_image = GUV_image.max(axis=0).squeeze()
        if MT_image.ndim ==3:
            MT_image = MT_image.max(axis=0).squeeze()

        cx_min = max(0, int(cx - diameter/2*1.5))
        cx_max = min(int(cx + diameter/2*1.5), GUV_image.shape[0]-1)
        cy_min = max(0, int(cy - diameter/2*1.5))
        cy_max = min(int(cy + diameter/2*1.5), GUV_image.shape[1]-1)

        mt_img_medianfiltered = median(MT_image, disk(3))
        mt_img_medianfiltered_smooth = gaussian(mt_img_medianfiltered, sigma, preserve_range=True)
        mt_img_normed = (mt_img_medianfiltered_smooth - mt_1p)/(mt_99p - mt_1p)
        mt_img_normed[mt_img_normed<0] = 0
        mt_img_normed[mt_img_normed>1] = 1

        guv_smooth = gaussian(GUV_image, sigma, preserve_range=True)
        guv_img_normed = (guv_smooth - guv_1p)/(guv_99p - guv_1p)
        guv_img_normed[guv_img_normed<0] = 0
        guv_img_normed[guv_img_normed>1] = 1    
        from scipy.ndimage import gaussian_filter

        blur1 = gaussian_filter(guv_img_normed, sigma=1)
        blur2 = gaussian_filter(guv_img_normed, sigma=3)
        dog = blur1 - blur2
        dog[dog<0]=0
        dog_image = dog/(0.25)               

        DOG_image_crop =  dog_image[cx_min:cx_max, cy_min:cy_max]
        GUV_image_crop =  guv_img_normed[cx_min:cx_max, cy_min:cy_max]
        MT_image_crop =  mt_img_normed[cx_min:cx_max, cy_min:cy_max]

        fig, ax = plt.subplots(3, 5, figsize=(13, 6),dpi=200)
        # plt.tight_layout()
        fig.suptitle(condition+', '+date+', '+ GUV_file_name)
        ax[0,0].imshow(guv_img_normed, clim=(0,1), cmap='gray')    
        ax[0,0].axis('off')
        ax[0,0].set_title('GUV')
        ax[1,0].imshow(mt_img_normed, clim=(0,1), cmap='gray')    
        ax[1,0].axis('off')
        ax[1,0].set_title('MT')

        ax[0,1].imshow(GUV_image_crop, clim=(0,1), cmap='gray')  
        ax[0,1].axis('off')
        ax[0,1].set_title('GUV')
        
        ax[1,1].imshow(MT_image_crop, clim=(0,1), cmap='gray')    
        ax[1,1].axis('off')
        ax[1,1].set_title('MT')


        ax[0,4].axis('off')
        category = str(row['Category']) if pd.notna(row['Category']) else "Unknown"
        ax[0, 4].set_title('Annotated as ' + category)
        
        ax[1,4].imshow(cellpose_crop_MT_image, clim=(0,1), cmap='gray')    
        ax[1,4].axis('off')
        ax[1,4].set_title('cellpose')

        ax[1,3].axis('off')
        ax[1,2].axis('off')
        ax[0,2].axis('off')
        ax[0,3].axis('off')

        ax[2,0].axis('off')
        ax[2,1].axis('off')
        ax[2,2].axis('off')
        ax[2,3].axis('off')
        ax[2,4].axis('off')

        GUV_image_crop_HOUGH = cv2.normalize(DOG_image_crop*255, None, 0, 255, cv2.NORM_MINMAX)
        GUV_image_crop_HOUGH = np.uint8(GUV_image_crop_HOUGH)

        ax[0,2].imshow(GUV_image_crop, clim=(0,1), cmap='gray')         
        ax[0,2].set_title('GUV')
        
        ax[1,2].imshow(MT_image_crop, clim=(0,1), cmap='gray')          
        ax[1,2].set_title('MT')

        circles = cv2.HoughCircles(
                GUV_image_crop_HOUGH, cv2.HOUGH_GRADIENT, dp=1.2, minDist=50, param1=15, param2=int(diameter/2*1.1), minRadius=int(diameter/2*0.7), maxRadius=int(diameter/2*1.1)
            )
        
        if circles is None or len(circles) == 0:
            print("empty: " +patch_tif_filename)
            circle_x = cx - cx_min
            circle_y = cy - cy_min
            circle_radius = diameter/2

            plot_circle = plt.Circle((circle_x + cy_min, circle_y + cx_min), circle_radius, fill=False, linewidth=1, color='red')                      
            ax[0,0].add_patch(plot_circle)            
            plot_circle = plt.Circle((circle_x + cy_min, circle_y + cx_min), circle_radius, fill=False, linewidth=1, color='red')                      
            ax[1,0].add_patch(plot_circle)            
            

            plot_circle = plt.Circle((circle_x, circle_y), circle_radius, fill=False, linewidth=2, color='red')                      
            ax[0,2].add_patch(plot_circle)            
            plot_circle = plt.Circle((circle_x, circle_y), circle_radius, fill=False, linewidth=2, color='red')                      
            ax[1,2].add_patch(plot_circle)
        else:
            circle_x = circles[0][0,0]+0
            circle_y = circles[0][0,1]+0
            circle_radius = circles[0][0,2]+0

            plot_circle = plt.Circle((circle_x + cy_min, circle_y + cx_min), circle_radius, fill=False, linewidth=1, color='blue')                      
            ax[0,0].add_patch(plot_circle)            
            plot_circle = plt.Circle((circle_x + cy_min, circle_y + cx_min), circle_radius, fill=False, linewidth=1, color='blue')                      
            ax[1,0].add_patch(plot_circle)      

            plot_circle = plt.Circle((circle_x, circle_y), circle_radius, fill=False, linewidth=2, color='blue')                      
            ax[0,2].add_patch(plot_circle)          
            plot_circle = plt.Circle((circle_x, circle_y), circle_radius, fill=False, linewidth=2, color='blue')                      
            ax[1,2].add_patch(plot_circle)

        GUV_square_image = disk_to_square(GUV_image_crop.astype(np.float32), cx=circle_x, cy=circle_y, r=circle_radius*0.9999, output_size=128)
        ax[0,3].imshow(GUV_square_image, clim=(0,1), cmap='gray')    
        ax[0,3].set_title('GUV wrap')

        MT_square_image = disk_to_square(MT_image_crop.astype(np.float32), cx=circle_x, cy=circle_y, r=circle_radius*0.95, output_size=128)
        ax[1,3].imshow(MT_square_image, clim=(0,1), cmap='gray')    
        ax[1,3].set_title('MT wrap')

        
        MT_rescale_circle_crop_image = crop_rescale_circle(MT_image_crop.astype(np.float32), cx=circle_x, cy=circle_y, r=circle_radius*0.90, output_size=128)
        ax[2,2].imshow(MT_rescale_circle_crop_image, clim=(0,1), cmap='gray')    
        ax[2,2].set_title('MT rescaled cricle')
        
        MT_circle_crop_image = crop_circle_detour(MT_image_crop.astype(np.float32), cx=circle_x, cy=circle_y, r=circle_radius*0.90, output_size=128)
        ax[2,3].imshow(MT_circle_crop_image, clim=(0,1), cmap='gray')    
        ax[2,3].set_title('MT cricle')

        
        display_image = MT_image_crop*255
        display_image[display_image>=255]=254

        obj_filename = condition+'-'+date+'-'+ GUV_file_name + '_cell'+str(cell_ID).zfill(2)+'_cellindex'+str(cell_index).zfill(4)+'.png'
        tifffile.imsave(os.path.join(annotated_circle_detection_png_folder, obj_filename),(display_image).astype(np.uint8))  
        
        obj_filename = condition+'-'+date+'-'+ GUV_file_name + '_cell'+str(cell_ID).zfill(2)+'_cellindex'+str(cell_index).zfill(4)+'.tif'
        tifffile.imwrite(os.path.join(annotated_circle_detection_tiff_folder, obj_filename),MT_image_crop.astype(np.float16))  
        
        obj_filename = condition+'-'+date+'-'+ GUV_file_name + '_cell'+str(cell_ID).zfill(2)+'_cellindex'+str(cell_index).zfill(4)+'.tif'
        tifffile.imwrite(os.path.join(annotated_circle_square_tiff_folder, obj_filename),MT_square_image.astype(np.float16))  
        
        obj_filename = condition+'-'+date+'-'+ GUV_file_name + '_cell'+str(cell_ID).zfill(2)+'_cellindex'+str(cell_index).zfill(4)+'.tif'
        tifffile.imwrite(os.path.join(annotated_MT_circle_crop_tiff_folder, obj_filename),MT_circle_crop_image.astype(np.float16))  
        
        obj_filename = condition+'-'+date+'-'+ GUV_file_name + '_cell'+str(cell_ID).zfill(2)+'_cellindex'+str(cell_index).zfill(4)+'.tif'
        tifffile.imwrite(os.path.join(annotated_MT_rescaled_circle_crop_tiff_folder, obj_filename),MT_rescale_circle_crop_image.astype(np.float16))  
        
        fig.savefig(os.path.join(annotated_circle_square_panel_folder, condition+'-'+date+'-'+GUV_file_name+ '_cell'+str(cell_ID).zfill(2)+'_cellindex'+str(cell_index).zfill(4) +'_circle_filter.png'))
        plt.close(fig) 

        counts = counts + 1



        # break      
        
       


empty: 1_10 Tau_Tubulin-Date 2-Image3_GUV.TIF_cell02_cellindex0018.tif
empty: 1_10 Tau_Tubulin-Date 2-Image4_GUV.TIF_cell01_cellindex0021.tif
empty: 1_10 Tau_Tubulin-Date 3-Image 7_GUV.TIF_cell03_cellindex0060.tif
empty: 1_10 Tau_Tubulin-Date 3-Image 7_GUV.TIF_cell07_cellindex0063.tif
empty: 1_10 Tau_Tubulin-Date 3-Image 8_GUV.TIF_cell14_cellindex0075.tif
empty: 1_10 Tau_Tubulin-Date 3-Image 8_GUV.TIF_cell17_cellindex0078.tif
empty: 1_10 Tau_Tubulin-Date 3-Image 9_GUV.TIF_cell14_cellindex0090.tif
empty: 1_10 Tau_Tubulin-Date 4-1 uM tau + mt encap GUV19_w1561.TIF_cell04_cellindex0095.tif
empty: 1_10 Tau_Tubulin-Date 4-1 uM tau + mt encap GUV73_w1561.TIF_cell07_cellindex0162.tif
empty: 1_10 Tau_Tubulin-Date 5-1 uM tau mt encap guv14_w1561.TIF_cell03_cellindex0175.tif
empty: 1_10 Tau_Tubulin-Date 5-1 uM tau mt encap guv26_w1561.TIF_cell04_cellindex0181.tif
empty: 1_10 Tau_Tubulin-Date 5-1 uM tau mt encap guv5_w1561.TIF_cell03_cellindex0205.tif
empty: 1_10 Tau_Tubulin-Date 5-1 uM tau mt en

In [11]:
annotation_csv.__len__

<bound method DataFrame.__len__ of                                               Filename Category
0    1_10 Tau_Tubulin-Date 1-Image10_GUV.TIF_cell02...    Patch
1    1_10 Tau_Tubulin-Date 1-Image3_GUV.TIF_cell03_...    Patch
2    1_10 Tau_Tubulin-Date 1-Image6_GUV.TIF_cell01_...  Network
3    1_10 Tau_Tubulin-Date 1-Image8_GUV.TIF_cell04_...  Nothing
4    1_10 Tau_Tubulin-Date 1-Image9_GUV.TIF_cell02_...  Network
..                                                 ...      ...
349  3_10 Tau_Tubulin-Date 5-2 uM tau mt encap guv1...    Patch
350  3_10 Tau_Tubulin-Date 5-2 uM tau mt encap guv8...    Patch
351  3_10 Tau_Tubulin-Date 5-2 uM tau mt encap guv8...    Patch
352  3_10 Tau_Tubulin-Date 5-2 uM tau mt encap guv8...    Patch
353  3_10 Tau_Tubulin-Date 5-2 uM tau mt encap guv9...    Patch

[354 rows x 2 columns]>

In [12]:
counts

354